### Tools
Models Can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are fearing of : 
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON Schema) 
2. A function or coroutine to execute

In [5]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

# response = model.invoke("why do I like girl?")
# response.content

In [ ]:
from langchain.tools import tool
# import requests

@tool
def getWeather(location: str) -> str:
    """Get current weather information for a city."""
    if "kolkata" in location:
        return "The weather in Kolkata is currently rainy."
    return "sunny"

model_with_tools = model.bind_tools([getWeather])

In [ ]:
response = model_with_tools.invoke("What's the weather in Dubai?")   #--->model with tools
print(response)
for tool_call in response.tool_calls:
    # tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Dubai. Use getWeather function.', 'tool_calls': [{'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'function': {'arguments': '{"location":"Dubai"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 127, 'total_tokens': 167, 'completion_time': 0.083790597, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.004881125, 'prompt_tokens_details': None, 'queue_time': 0.402363824, 'total_time': 0.088671722}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0baab-671d-70c1-813e-9def87b7e1fa-0' tool_calls=[{'name': 'getWeather', 'args': {'location': 'Dubai'}, 'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 12

In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    if "kolkata" in location.lower():
        return "The weather in Kolkata is currently rainy."
    return "sunny"

# Create the agent with the weather tool
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_weather]
)


In [3]:
response = agent.invoke({
    "messages": [("user", "What's the weather in Dubai?")]
})
print(response)
for msg in response["messages"]:
    tool_calls = getattr(msg, "tool_calls", None) or []
    for tool_call in tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")

{'messages': [HumanMessage(content="What's the weather in Dubai?", additional_kwargs={}, response_metadata={}, id='f440b3c1-8cdf-4833-b12b-3ed5b215ea39'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Dubai"}'}, '__gemini_function_call_thought_signatures__': {'call_83101': 'EnEKbwFpFH0TA4YbIe8JBTKW54BfWjXZaVrbz29XuZkxHe+Kmun7/A24N8OJrQ4gl5mlADuWWmgzEJyDNsEPY7iKoMXBEcENZV3MbXp982R7DC1Y/rsJX8e6gzFZQiZk6DzAY4P9V+6yQVlmgkOZ9FG8FQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d671-69ee-79b1-bfe2-21cfc26700fe-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Dubai'}, 'id': 'call_83101', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 16, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='sunny', name='get_weather', id='

#### Tool execution Loop

In [ ]:
'''Model generates tool call'''
messages = [{"role" : "user", "content": "What's the weather in kolkata ?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

'''# Execute tools and collect result'''
for tool_call in ai_msg.tool_calls:
    '''# execute the tool with arguments'''
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

'''if you want the raw content'''
# print(messages[-1].content)
# print(tool_result.content)
'''# pass result back to model for final response'''
'''why feed it back to llm ? can't you use temperature and top_p to generate response in one go?'''
final_response = model_with_tools.invoke(messages)
print(final_response.text)

Sure thing! Right now, Kolkata is experiencing **rainy** weather. 🌧️ If you’re heading out, you might want to bring an umbrella or a raincoat. Stay dry!
